In [1]:
import os
import sys
import time
import pandas as pd
import numpy  as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import warnings
warnings.filterwarnings('ignore')

module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path:
    sys.path.append(module_path)

from sklearn.model_selection import train_test_split
from sklearn.model_selection import KFold
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import roc_auc_score
from xgboost import XGBClassifier
from xgboost import plot_importance
from lightgbm import LGBMClassifier
from sklearn.ensemble import RandomForestClassifier
import lightgbm as lgbm

from hyperopt import hp
from hyperopt import fmin, tpe, Trials, STATUS_OK
from hyperopt import space_eval

from utils import user_utils
from utils import preprocessing

In [2]:
# data loading
train_df ,test_df= preprocessing.load_data()
test_df = test_df.drop(["ID"], axis=1)

In [3]:
X_features, y_target = preprocessing.split_features_target(train_df)

In [4]:
X_features['var3'] = X_features['var3'].replace(-999999, 2)

In [ ]:
# 컬럼 삭제 ,결측치 처리 및 스탠다드 스케일처
# from common.preprocessing import load_data
scaled_X_train, scaled_X_test, scaler = preprocessing.scale_data(X_features, test_df)

In [6]:
X_train, X_val, y_train, y_val = preprocessing.data_split(scaled_X_train, y_target)

In [7]:
# --- 2. Hyperparameter 탐색 공간 정의 ---

# XGBoost 탐색 공간
xgb_search_space = {
    'max_depth': hp.quniform('max_depth', 5, 15, 1),
    'learning_rate': hp.uniform('learning_rate', 0.01, 0.2),
    'n_estimators': hp.quniform('n_estimators', 100, 1000, 10),
    'min_child_weight': hp.quniform('min_child_weight', 1, 6, 1),
    'subsample': hp.uniform('subsample', 0.7, 1.0),
    'colsample_bytree': hp.uniform('colsample_bytree', 0.7, 1.0),
    'gamma': hp.uniform('gamma', 0, 0.5)
}

# LightGBM 탐색 공간
lgbm_search_space = {
    'num_leaves': hp.quniform('num_leaves', 32, 128, 1),
    'learning_rate': hp.uniform('learning_rate', 0.01, 0.2),
    'n_estimators': hp.quniform('n_estimators', 100, 1000, 10),
    'subsample': hp.uniform('subsample', 0.7, 1.0),
    'colsample_bytree': hp.uniform('colsample_bytree', 0.7, 1.0),
    'reg_alpha': hp.uniform('reg_alpha', 0, 1),
    'reg_lambda': hp.uniform('reg_lambda', 0, 1),
}

# RandomForest 탐색 공간
rf_search_space = {
    'n_estimators': hp.quniform('n_estimators', 100, 500, 10),
    'max_depth': hp.quniform('max_depth', 10, 30, 1),
    'min_samples_leaf': hp.quniform('min_samples_leaf', 1, 8, 1),
    'min_samples_split': hp.quniform('min_samples_split', 2, 12, 1)
}


In [8]:
def xgb_objective(params):
    """XGBoost 목적 함수"""
    params['max_depth'] = int(params['max_depth'])
    params['n_estimators'] = int(params['n_estimators'])

    xgb = XGBClassifier(
        **params,
        eval_metric = 'auc',
        use_label_encoder = False,
        early_stopping_rounds = 30,
        random_state = 23
    )

    xgb.fit(
        X_train, y_train,
        eval_set = [(X_val, y_val)],
        verbose = False
    )

    roc_auc = roc_auc_score(y_val, xgb.predict_proba(X_val)[:, 1])
    
    return {'loss': -roc_auc, 'status': STATUS_OK}


def lgbm_objective(params):
    """LightGBM 목적 함수"""
    params['num_leaves'] = int(params['num_leaves'])
    params['n_estimators'] = int(params['n_estimators'])
    
    lgbm = LGBMClassifier(
        **params,
        random_state = 23,
        early_stopping_rounds = 30,
        verbose = -1
    )
    
    lgbm.fit(
        X_train, y_train,
        eval_set = [(X_val, y_val)]
    )

    roc_auc = roc_auc_score(y_val, lgbm.predict_proba(X_val)[:, 1])

    return {'loss': -roc_auc, 'status': STATUS_OK}

def rf_objective(params):
    """RandomForest 목적 함수"""
    params['n_estimators'] = int(params['n_estimators'])
    params['max_depth'] = int(params['max_depth'])
    params['min_samples_leaf'] = int(params['min_samples_leaf'])
    params['min_samples_split'] = int(params['min_samples_split'])

    rf = RandomForestClassifier(
        **params,
        random_state = 23,
        n_jobs = -1
    )

    rf.fit(X_train, y_train)
    roc_auc = roc_auc_score(y_val, rf.predict_proba(X_val)[:, 1])

    return {'loss': -roc_auc, 'status': STATUS_OK}


In [9]:
MAX_EVALS = 50 

# XGBoost 튜닝
print("\n--- XGBoost Hyperparameter Tuning ---")
start_time = time.time()
xgb_trials = Trials()
best_xgb = fmin(
    fn = xgb_objective,
    space = xgb_search_space,
    algo = tpe.suggest,
    max_evals = MAX_EVALS,
    trials = xgb_trials,
    rstate = np.random.default_rng(seed=23)
)
end_time = time.time()
print(f"튜닝 시간: {end_time - start_time:.2f}초")
print("최적 하이퍼파라미터:", best_xgb)
print("최고 AUC:", -xgb_trials.best_trial['result']['loss'])

# 실제 하이퍼파라미터 딕셔너리로 변환
best_params = space_eval(xgb_search_space, best_xgb)

# 정수형 파라미터 변환
best_params['max_depth'] = int(best_params['max_depth'])
best_params['n_estimators'] = int(best_params['n_estimators'])

# 최적 파라미터로 모델 생성 및 학습
xgb_best = XGBClassifier(
    **best_params,
    eval_metric='auc',
    use_label_encoder=False,
    random_state=23
)

user_utils.get_model_train_eval(xgb_best,'xgb_HyperOpt_ejm',X_train, X_val, y_train, y_val)




--- XGBoost Hyperparameter Tuning ---
  0%|          | 0/50 [00:00<?, ?trial/s, best loss=?]

100%|██████████| 50/50 [01:50<00:00,  2.21s/trial, best loss: -0.8526307778345569]
튜닝 시간: 110.33초
최적 하이퍼파라미터: {'colsample_bytree': np.float64(0.8828333771389649), 'gamma': np.float64(0.058408742978283044), 'learning_rate': np.float64(0.12721361071578832), 'max_depth': np.float64(6.0), 'min_child_weight': np.float64(2.0), 'n_estimators': np.float64(320.0), 'subsample': np.float64(0.845580758889997)}
최고 AUC: 0.8526307778345569
✓ 모델 저장 완료: models\xgb_HyperOpt_ejm.pkl
  파일 크기: 0.82 MB
folder = c:\big20\git\big20-ML-project2-team3\SantanderCS\results
AUC: 0.8324, 정확도: 0.9599, 정밀도: 0.3333, 재현율: 0.0116, F1: 0.0225
오차행렬:
[[14588    14]
 [  595     7]]
실행 시간: 4.026152849197388


In [13]:
# LightGBM 튜닝
print("\n--- LightGBM Hyperparameter Tuning ---")
start_time = time.time()
lgbm_trials = Trials()
best_lgbm = fmin(
    fn = lgbm_objective,
    space = lgbm_search_space,
    algo = tpe.suggest,
    max_evals = MAX_EVALS,
    trials = lgbm_trials,
    rstate = np.random.default_rng(seed=23)
)
end_time = time.time()
print(f"튜닝 시간: {end_time - start_time:.2f}초")
print("최적 하이퍼파라미터:", best_lgbm)
print("최고 AUC:", -lgbm_trials.best_trial['result']['loss'])

best_lgb = space_eval(lgbm_search_space, best_lgbm)
best_lgb['num_leaves'] = int(best_lgb['num_leaves'])
best_lgb['n_estimators'] = int(best_lgb['n_estimators'])

lgbm_best = LGBMClassifier(
    **best_lgb,
    random_state=23
)

user_utils.get_model_train_eval(lgbm_best,'lgbm_HyperOpt_ejm',X_train, X_val, y_train, y_val)



--- LightGBM Hyperparameter Tuning ---
100%|██████████| 50/50 [01:34<00:00,  1.88s/trial, best loss: -0.8528572179390163]
튜닝 시간: 94.20초
최적 하이퍼파라미터: {'colsample_bytree': np.float64(0.7342406115797382), 'learning_rate': np.float64(0.02751826185901235), 'n_estimators': np.float64(660.0), 'num_leaves': np.float64(42.0), 'reg_alpha': np.float64(0.6127977982911577), 'reg_lambda': np.float64(0.1262561992869149), 'subsample': np.float64(0.973776538266153)}
최고 AUC: 0.8528572179390163
✓ 모델 저장 완료: models\lgbm_HyperOpt_ejm.pkl
  파일 크기: 2.89 MB
folder = c:\big20\git\big20-ML-project2-team3\SantanderCS\results
AUC: 0.8438, 정확도: 0.9605, 정밀도: 0.6000, 재현율: 0.0050, F1: 0.0099
오차행렬:
[[14600     2]
 [  599     3]]
실행 시간: 4.948836803436279


In [15]:
# RandomForest 튜닝
print("\n--- RandomForest Hyperparameter Tuning ---")
start_time = time.time()
rf_trials = Trials()
best_rf = fmin(
    fn = rf_objective,
    space = rf_search_space,
    algo = tpe.suggest,
    max_evals = MAX_EVALS,
    trials = rf_trials,
    rstate = np.random.default_rng(seed=23)
)
end_time = time.time()
print(f"튜닝 시간: {end_time - start_time:.2f}초")
print("최적 하이퍼파라미터:", best_rf)
print("최고 AUC:", -rf_trials.best_trial['result']['loss'])

best_rf = space_eval(rf_search_space, best_rf)

best_rf['n_estimators'] = int(best_rf['n_estimators'])
best_rf['max_depth'] = int(best_rf['max_depth'])
best_rf['min_samples_leaf'] = int(best_rf['min_samples_leaf'])
best_rf['min_samples_split'] = int(best_rf['min_samples_split'])

rf_best = RandomForestClassifier(
    **best_rf,
    random_state=23,
    n_jobs=-1
)

user_utils.get_model_train_eval(rf_best,'rf_HyperOpt_ejm',X_train, X_val, y_train, y_val)

print("\nHyperparameter 튜닝이 완료되었습니다.")


--- RandomForest Hyperparameter Tuning ---
100%|██████████| 50/50 [11:23<00:00, 13.67s/trial, best loss: -0.836193990628872] 
튜닝 시간: 683.38초
최적 하이퍼파라미터: {'max_depth': np.float64(25.0), 'min_samples_leaf': np.float64(1.0), 'min_samples_split': np.float64(7.0), 'n_estimators': np.float64(390.0)}
최고 AUC: 0.836193990628872
✓ 모델 저장 완료: models\rf_HyperOpt_ejm.pkl
  파일 크기: 65.74 MB
folder = c:\big20\git\big20-ML-project2-team3\SantanderCS\results
AUC: 0.8362, 정확도: 0.9605, 정밀도: 0.6000, 재현율: 0.0050, F1: 0.0099
오차행렬:
[[14600     2]
 [  599     3]]
실행 시간: 18.158472299575806

Hyperparameter 튜닝이 완료되었습니다.
